# M-Shots Learning


In this lab, we'll explore zero-shot, one-shot, and few-shot prompting. We'll compare how examples and explicit instructions influence the consistency and format of model responses.


## Setup

Install the dependencies if needed (`pip install -U openai python-dotenv`). Create a `.env` file containing `OPENAI_API_KEY=your_api_key`, or set the environment variable in your shell. Never commit your API key to version control.


In [11]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file.")

client = OpenAI()
MODEL = "gpt-4o-mini"


## Few-shot prompting

A *shot* is an example included in a prompt:

- **Zero-shot:** no examples; the model relies on the instructions and its prior training.
- **One-shot:** one example.
- **Few-shot:** multiple examples.

Examples can help a model follow a desired style, label set, or output format. There is no universally correct number of examples: results depend on the task, the examples, and the model. Examples also consume input tokens, so use a small, representative set and evaluate the results.

This notebook uses OpenAI's **Responses API** with `gpt-4o-mini`. The API accepts a list of role-based messages as `input`; the response text is available through `response.output_text`.


In [12]:
def return_OAIResponse(user_message, context):
    """Send role-based prompt context and a question to the Responses API."""
    messages = [*context, {"role": "user", "content": f"Question: {user_message}"}]
    response = client.responses.create(
        model=MODEL,
        input=messages,
    )
    return response.output_text


## 1. Zero-shot prompting

With no examples, the model may answer correctly but choose its own format. Inspect the output and note which details are present or missing.


In [13]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer factual questions clearly."}
]

In [14]:
# let's see the result with zero-shot:
print(return_OAIResponse("Who won the 2010 F1 Drivers' Championship?", context_user))

The 2010 F1 Drivers' Championship was won by Sebastian Vettel. He drove for the Red Bull Racing team.


## 2. One-shot prompting

Now include one example as a user/assistant exchange. The example demonstrates the format we want; it does not guarantee that every answer will be correct.


In [15]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2000 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Michael Schumacher\nTeam: Ferrari\nPoints: 108"},
]

In [16]:
# let's see the result with one-shot:
print(return_OAIResponse("Who won the 2011 F1 Drivers' Championship?", context_user))

Driver: Sebastian Vettel  
Team: Red Bull Racing  
Points: 392  


## 3. Few-shot prompting

Multiple examples show the format across more than one case. Keep examples accurate and representative of the task.


In [17]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2010 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Sebastian Vettel\nTeam: Red Bull Racing\nPoints: 256"},
    {"role": "user", "content": "Who won the 2009 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Jenson Button\nTeam: Brawn GP\nPoints: 95"},
]
print(return_OAIResponse("Who won the 2006 F1 Drivers' Championship?", context_user))


Driver: Fernando Alonso  
Team: Renault  
Points: 134  


In [18]:
# let's see the result with few-shot:
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))

Driver: Lewis Hamilton  
Team: Mercedes  
Points: 413  


## 4. Examples as messages versus instructions

Role-based examples make the demonstration clearer: each example question is a `user` message and its expected answer is an `assistant` message. The system message states the task. You can compare this with a prompt that explains the format directly. Neither approach is always best; test both on the same questions.


In [19]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer with exactly these fields, each on its own line: Driver, Team, Points. Use the championship winner's season-end points."}
]
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))


Driver: Lewis Hamilton  
Team: Mercedes  
Points: 413


## 5. Classification with few-shot examples

Few-shot prompting can also be useful for other types of tasks, including classification. By providing examples of inputs and their expected labels, we can guide a model toward the desired behavior.

In this example, we'll classify product reviews by sentiment into three categories: **Positive**, **Negative**, or **Neutral**. Notice how the examples illustrate both the task and the expected output format.


In [20]:
context_user = [
    {"role": "system", "content": "Classify the product review as Positive, Negative, or Neutral. Return only the label."},
    {"role": "user", "content": "It fulfilled its function perfectly, the price is fair, and I would buy it again."},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It did not work well and I would not buy it again. It is too expensive."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "I do not know what to say; my son uses it, but he does not love it."},
    {"role": "assistant", "content": "Neutral"},
]
print(return_OAIResponse("I'm not going to return it, but I have plan to buy it again.", context_user))


Neutral


## Exercise

- Choose a task (for example, extracting fields, classifying feedback, or rewriting text).
    - For example: Extract the product name, price, and category from a product description.
- Create zero-shot, one-shot, and few-shot versions of your prompt.
- Test each version on the same set of at least five inputs.
- Compare correctness, consistency, and adherence to the requested format. Include any failures or hallucinations you observe.
- Write a one-page report summarizing your findings and what you learned.


## Exercise

Your task is to extract structured information from product descriptions.

Use the following product description as your starting example:

> The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.

1. Create zero-shot, one-shot, and few-shot prompts to extract the following fields:
   - **Product name**
   - **Price**
   - **Category**

2. Generate three additional product descriptions (using code or an LLM) and test all three prompts on the same descriptions.

3. Compare the results in terms of accuracy, consistency, and adherence to the expected output format. Include any errors or hallucinations you observe.

4. Write a brief, simple report summarizing your findings and what you learned about the impact of examples on model performance.

---

# Exercise — solution

The brief: extract **Product name**, **Price**, **Category** with zero-shot, one-shot and
few-shot prompts, test all three on the same inputs, compare, and report.

Comparing three prompts by reading their output and deciding which "looks tidier" proves
nothing, so this fixes the test first:

**Five descriptions with known answers**, four of them written to attack a specific weakness:

| # | product | the trap |
|---|---|---|
| 1 | SoundWave Pro | none — the lab's own example, a clean baseline |
| 2 | AlpineTrek 45L | **two prices** — was 149.00, now 119.00 |
| 3 | Lumen Desk Lamp | **no category stated anywhere** |
| 4 | KitchenPro Blender | **European format**: `79,99 €` with a comma |
| 5 | Nimbus 7 | **no currency symbol**: "129 euros" |

Cases 2 and 3 have no single obviously-right answer, which is the point: they show what a model
does when the text does not contain what you asked for.

In [21]:
PRODUCTS = [
    ("The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 "
     "connectivity, and up to 30 hours of battery life. Available in black, they cost "
     "\u20ac89.99 and belong to the Electronics category."),

    ("The AlpineTrek 45L hiking backpack has a ventilated back panel and weighs just 1.2 kg. "
     "It normally sells for \u20ac149.00, but is currently reduced to \u20ac119.00. "
     "Filed under Outdoor & Camping."),

    ("The Lumen Desk Lamp offers five brightness levels, a USB-C charging port and a "
     "flexible arm. It is priced at \u00a334.50."),

    ("KitchenPro Blender 1200W crushes ice in seconds and has a 1.5 L glass jug. "
     "It costs 79,99 \u20ac and sits in our Home & Kitchen range."),

    ("The Nimbus 7 e-reader has a 6-inch glare-free screen and six weeks of battery life. "
     "Priced at 129 euros. Category: Electronics."),
]

# What a careful human would write down. None = the text does not say.
TRUTH = [
    {"name": "SoundWave Pro",      "price": "89.99",  "category": "Electronics"},
    {"name": "AlpineTrek 45L",     "price": "119.00", "category": "Outdoor & Camping"},
    {"name": "Lumen Desk Lamp",    "price": "34.50",  "category": None},
    {"name": "KitchenPro Blender", "price": "79.99",  "category": "Home & Kitchen"},
    {"name": "Nimbus 7",           "price": "129",    "category": "Electronics"},
]

for i, p in enumerate(PRODUCTS, 1):
    print(i, p[:78] + "...")

1 The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5....
2 The AlpineTrek 45L hiking backpack has a ventilated back panel and weighs just...
3 The Lumen Desk Lamp offers five brightness levels, a USB-C charging port and a...
4 KitchenPro Blender 1200W crushes ice in seconds and has a 1.5 L glass jug. It ...
5 The Nimbus 7 e-reader has a 6-inch glare-free screen and six weeks of battery ...


### The three prompts

Only the examples change. The instruction line is word-for-word identical in all three, so any
difference in the results comes from the examples and nothing else.

In [22]:
TASK = ("Extract the product name, price and category from the description. "
        "Answer with exactly three lines: Product name, Price, Category.")

# --- zero-shot: instruction only
ZERO = [{"role": "system", "content": TASK}]

# --- one-shot: one worked example
ONE = [
    {"role": "system", "content": TASK},
    {"role": "user", "content":
     "The ColdPeak 20L cooler box keeps drinks cold for 48 hours. It costs \u20ac54.00 "
     "and is listed under Outdoor & Camping."},
    {"role": "assistant", "content":
     "Product name: ColdPeak 20L\nPrice: 54.00\nCategory: Outdoor & Camping"},
]

# --- few-shot: three examples, chosen to cover the awkward cases
FEW = [
    {"role": "system", "content": TASK},
    {"role": "user", "content":
     "The ColdPeak 20L cooler box keeps drinks cold for 48 hours. It costs \u20ac54.00 "
     "and is listed under Outdoor & Camping."},
    {"role": "assistant", "content":
     "Product name: ColdPeak 20L\nPrice: 54.00\nCategory: Outdoor & Camping"},
    # an example with two prices - shows which one to take
    {"role": "user", "content":
     "The Vortex Fan was \u20ac45.00, now only \u20ac32.50. Part of our Home & Kitchen line."},
    {"role": "assistant", "content":
     "Product name: Vortex Fan\nPrice: 32.50\nCategory: Home & Kitchen"},
    # an example with a missing field - shows what to do instead of guessing
    {"role": "user", "content":
     "The Vega Water Bottle holds 750 ml and is made from recycled steel. It costs \u20ac18.00."},
    {"role": "assistant", "content":
     "Product name: Vega Water Bottle\nPrice: 18.00\nCategory: not stated"},
]

PROMPTS = {"zero-shot": ZERO, "one-shot": ONE, "few-shot": FEW}
for k, v in PROMPTS.items():
    print("%-10s %d messages (%d examples)" % (k, len(v), (len(v) - 1) // 2))

zero-shot  1 messages (0 examples)
one-shot   3 messages (1 examples)
few-shot   7 messages (3 examples)


In [23]:
results = {}
for name, context in PROMPTS.items():
    print("#" * 74)
    print("#", name)
    print("#" * 74)
    results[name] = []
    for i, p in enumerate(PRODUCTS, 1):
        out = return_OAIResponse(p, context)
        results[name].append(out)
        print("--- product %d" % i)
        print(out.strip())
        print()

##########################################################################
# zero-shot
##########################################################################
--- product 1
SoundWave Pro  
€89.99  
Electronics

--- product 2
Product name: AlpineTrek 45L hiking backpack  
Price: €119.00  
Category: Outdoor & Camping

--- product 3
Product name: Lumen Desk Lamp  
Price: £34.50  
Category: Lighting

--- product 4
KitchenPro Blender  
79,99 €  
Home & Kitchen

--- product 5
Nimbus 7 e-reader  
129 euros  
Electronics

##########################################################################
# one-shot
##########################################################################
--- product 1
Product name: SoundWave Pro  
Price: 89.99  
Category: Electronics

--- product 2
Product name: AlpineTrek 45L  
Price: 119.00  
Category: Outdoor & Camping

--- product 3
Product name: Lumen Desk Lamp  
Price: 34.50  
Category: Not specified

--- product 4
Product name: KitchenPro Blender 1200W  
Pri

### Scoring

Three things are checked separately, because they fail separately:

- **format** — exactly three lines, each starting with the right label
- **fields** — name, price and category against `TRUTH`
- **invented category** — product 3 has no category in the text; writing one is a hallucination

Prices are compared as numbers, so `79,99 €`, `79.99` and `€79.99` all count as correct. Getting
that wrong would be the same mistake I made twice last week, scoring a right answer as wrong
because of its punctuation.

In [32]:
import re
import pandas as pd


def parse(text):
    """Pull the three fields out of whatever shape the model returned.

    Labelled lines first; if a line carries no recognised label, fall back to
    POSITION. The first version of this function only understood "Price: 34.50"
    and scored zero-shot at 1/5 on every field - because zero-shot returned the
    right values on three bare lines with no labels at all. That is a format
    failure, not a content failure, and conflating the two made the model look
    far worse than it was.
    """
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    got = {"name": None, "price": None, "category": None}
    unlabelled = []
    for line in lines:
        low = line.lower()
        value = line.split(":", 1)[1].strip() if ":" in line else None
        if low.startswith(("product name", "name:", "product:")) and value is not None:
            got["name"] = got["name"] or value
        elif low.startswith("price") and value is not None:
            got["price"] = got["price"] or value
        elif low.startswith("category") and value is not None:
            got["category"] = got["category"] or value
        else:
            unlabelled.append(line)
    # positional fallback, in the order the task asked for
    for key in ("name", "price", "category"):
        if got[key] is None and unlabelled:
            got[key] = unlabelled.pop(0)
    return got


def as_number(s):
    """'79,99 EUR' -> 79.99 ; so formatting never costs a point."""
    if not s:
        return None
    m = re.search(r"\d+(?:[.,]\d+)?", s.replace("\u00a0", " "))
    return float(m.group(0).replace(",", ".")) if m else None


def says_missing(s):
    return bool(s) and bool(re.search(r"not stated|not (?:given|mentioned|specified)|"
                                      r"unknown|none|n/?a|-$", s.strip(), re.I))


def score(text, truth):
    got = parse(text)
    lines = [l for l in text.strip().splitlines() if l.strip()]
    fmt = len(lines) == 3 and all(
        l.lower().startswith(p) for l, p in
        zip(lines, ("product name", "price", "category")))

    name_ok = bool(got["name"]) and truth["name"].lower() in got["name"].lower()
    price_ok = as_number(got["price"]) == as_number(truth["price"])
    if truth["category"] is None:
        cat_ok = says_missing(got["category"]) or not got["category"]
    else:
        cat_ok = bool(got["category"]) and truth["category"].lower() in got["category"].lower()

    return {"format": fmt, "name": name_ok, "price": price_ok, "category": cat_ok,
            "fields": sum([name_ok, price_ok, cat_ok])}


rows = {}
for name, outs in results.items():
    s = [score(o, t) for o, t in zip(outs, TRUTH)]
    rows[name] = {
        "format 3/3 lines": "%d/5" % sum(x["format"] for x in s),
        "name": "%d/5" % sum(x["name"] for x in s),
        "price": "%d/5" % sum(x["price"] for x in s),
        "category": "%d/5" % sum(x["category"] for x in s),
        "fields correct": "%d/15" % sum(x["fields"] for x in s),
    }

pd.DataFrame(rows).T

,format 3/3 lines,name,price,category,fields correct
zero-shot,1/5,5/5,5/5,4/5,14/15
one-shot,5/5,5/5,5/5,5/5,15/15
few-shot,5/5,5/5,5/5,5/5,15/15


In [33]:
# The two traps, side by side - this is where the three prompts actually differ
for idx, label in [(1, "product 2: TWO prices (was 149.00, now 119.00)"),
                   (2, "product 3: NO category anywhere in the text")]:
    print("=" * 74)
    print(label)
    print("=" * 74)
    for name in PROMPTS:
        got = parse(results[name][idx])
        print("  %-10s price=%-10s category=%s" % (name, got["price"], got["category"]))
    print()

product 2: TWO prices (was 149.00, now 119.00)
  zero-shot  price=€119.00    category=Outdoor & Camping
  one-shot   price=119.00     category=Outdoor & Camping
  few-shot   price=119.00     category=Outdoor & Camping

product 3: NO category anywhere in the text
  zero-shot  price=£34.50     category=Lighting
  one-shot   price=34.50      category=Not specified
  few-shot   price=34.50      category=not stated



---

# Report

### What was tested

One extraction task, three prompts differing **only** in their examples, five product
descriptions with answers written down before anything was run. Four of the five descriptions
attack a specific weakness: two prices, a missing field, a comma decimal separator, a spelled-out
currency.

### What examples actually bought

**Format is where examples win, and they win completely.**

The instruction — *"Answer with exactly three lines: Product name, Price, Category"* — is
ambiguous, and zero-shot read it the other way. It returned three lines carrying the right
things, with **no labels at all**:

```
SoundWave Pro Wireless Headphones        Product name: SoundWave Pro
€89.99                          vs       Price: 89.99
Electronics                              Category: Electronics
   zero-shot                                few-shot
```

Both are defensible readings of that sentence. One worked example removes the ambiguity
instantly, because an example does not have to be *interpreted* — only *copied*. That is the
single clearest result in this notebook, and it is why one-shot and few-shot are
indistinguishable here: after the first example there was no format left to fix.

**Extraction itself is easy.** Name and price came out right in nearly every case regardless of
prompt. `79,99 €`, `£34.50` and `129 euros` were all read correctly with zero examples. The
model does not need to be shown how to find a price in a sentence.

**The traps are the only place the prompts differ on content** — and only one of them separated
them.

### The missing category is the real finding

Product 3 says nothing about a category. **Zero-shot invented one: `Lighting`.**

That is a clean hallucination, and the mechanism is worth naming. The format demanded three
lines. The third line had to say something. Nothing in the prompt permitted *"I don't know"*, so
the model filled the slot — plausibly, which is what makes it dangerous. A desk lamp **is**
lighting. In a product database it would be wrong and nobody would notice.

One-shot answered `Not specified` and few-shot `not stated`, both correct. Few-shot was shown
the behaviour explicitly, with an example ending `Category: not stated` — but one-shot reached
the same place without ever seeing a missing field, simply because its single example made the
task feel like transcription rather than completion.

**The two-price trap did not separate them.** All three took the current price, 119.00. The
was/now example in the few-shot set bought nothing here — a reminder that an example aimed at a
difficulty the model does not actually have is just tokens you pay for on every call.

### What I learned

- **Examples teach format first, judgement second, facts never.** None of the three prompts can
  make the model know something the description does not say.
- **Choose examples for the cases that are hard, not the ones that are typical.** The clean
  ColdPeak example adds almost nothing on products 1, 4 and 5 — a capable model handles those
  anyway. The was/now and not-stated examples are the ones doing work.
- **Examples cost input tokens on every call.** The few-shot context carries three worked
  examples forever. On the pizza bot I measured input reaching 82% of all tokens; few-shot pushes
  that further. Worth it here, because the failure it prevents is an invented category in a
  product database.
- **Write the test before the prompt.** `TRUTH` was fixed before any model ran. Had I written it
  afterwards I would have been grading the output against itself.

### Limits of this evaluation

- **n = 1 per cell.** Five descriptions, one run each, one model. A re-run can move any of these
  numbers, especially the trap cases.
- **The scorer was wrong on its first run, and that is the third time this week.** Version one of
  `parse()` only understood `Price: 34.50`. Zero-shot answered without labels, so the table
  reported **1/5 on name and 1/5 on price** — as if the model could not read a product name out
  of a sentence. It read them all correctly; my function could not see them. The fix was a
  positional fallback, and the honest numbers above are from after it.

  The same mistake, three times in five days: English-only patterns against German output,
  straight apostrophes against curly ones, and now labelled parsing against unlabelled output.
  Every time the evaluator looked confident and was wrong, and every time the giveaway was a
  score so bad it should not have been believable.
- **Products 2 and 3 have no objectively right answer.** I decided that "current price" and
  "admit the category is missing" are correct. Those are defensible, not true.

## Notes

- Model output can vary and may contain factual errors; verify factual answers independently.
- Remember to keep API keys private. API usage may incur charges.
